In [ ]:
# CELL 1: Install all required packages
!pip install -q fastapi uvicorn PyPDF2 python-docx python-multipart nest-asyncio
!pip install -q spacy textblob scikit-learn pandas numpy
!pip install -q transformers torch sentence-transformers

# Download spaCy model
!python -m spacy download en_core_web_sm

# Download NLTK data
import nltk
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt', quiet=True)

print("✅ All packages installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 47.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ All packages installed successfully!


In [ ]:
# CELL 2: Create enhanced legal_app.py with CSV & JSON support
%%writefile legal_app.py
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Dict, Optional
import uuid
import PyPDF2
import docx
import os
import json
import re
import csv
import io
import pandas as pd
from collections import Counter
from datetime import datetime

# ML imports
import spacy
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load spaCy model
try:
    nlp = spacy.load("en_core_web_sm")
except:
    import subprocess
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

app = FastAPI(title="Enhanced Legal Document Analyzer", version="3.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

documents = {}
analysis_cache = {}

# Legal clause patterns with weights
CLAUSE_PATTERNS = {
    'Termination': {
        'keywords': ['terminate', 'termination', 'breach', 'default', 'cancel', 'rescind', 'void'],
        'weight': 1.0,
        'risk_level': 'medium'
    },
    'Liability': {
        'keywords': ['liability', 'indemnify', 'indemnification', 'damages', 'responsible', 'hold harmless'],
        'weight': 1.0,
        'risk_level': 'high'
    },
    'Confidentiality': {
        'keywords': ['confidential', 'nondisclosure', 'nda', 'private', 'proprietary', 'trade secret'],
        'weight': 0.9,
        'risk_level': 'medium'
    },
    'Payment': {
        'keywords': ['payment', 'pay', 'fee', 'invoice', 'compensation', 'price', 'cost', 'charge'],
        'weight': 0.8,
        'risk_level': 'low'
    },
    'Jurisdiction': {
        'keywords': ['jurisdiction', 'governing law', 'venue', 'court', 'arbitration', 'dispute resolution'],
        'weight': 0.8,
        'risk_level': 'medium'
    },
    'Warranty': {
        'keywords': ['warranty', 'guarantee', 'represent', 'as is', 'merchantability', 'fitness'],
        'weight': 0.9,
        'risk_level': 'high'
    },
    'Indemnity': {
        'keywords': ['indemnify', 'indemnification', 'hold harmless', 'defend', 'reimburse'],
        'weight': 1.0,
        'risk_level': 'high'
    },
    'Force Majeure': {
        'keywords': ['force majeure', 'act of god', 'unforeseeable', 'beyond control', 'natural disaster'],
        'weight': 0.7,
        'risk_level': 'low'
    }
}

# Risk keywords with scores
RISK_KEYWORDS = {
    'unlimited liability': 0.95,
    'no warranty': 0.90,
    'as is': 0.85,
    'no liability': 0.90,
    'indemnify': 0.70,
    'breach': 0.65,
    'termination for convenience': 0.60,
    'liquidated damages': 0.75,
    'exclusive remedy': 0.50,
    'binding effect': 0.30
}

class QueryRequest(BaseModel):
    question: str
    top_k: int = 3

class DocumentAnalysis(BaseModel):
    doc_id: str
    filename: str
    summary: str
    clauses: List[Dict]
    entities: List[Dict]
    risks: List[Dict]
    sentiment: Dict
    keywords: Dict
    readability_score: float
    word_count: int
    confidence_scores: Dict

# ============ NEW: CSV EXTRACTION FUNCTION ============
def extract_text_from_csv(file_path: str) -> Dict:
    """Extract text and data from CSV file"""
    try:
        df = pd.read_csv(file_path)

        # Store structured data info
        structured_info = {
            'type': 'csv',
            'rows': len(df),
            'columns': df.columns.tolist(),
            'data_types': df.dtypes.astype(str).to_dict()
        }

        # Convert to text for NLP analysis
        text_parts = []

        # Add file summary
        text_parts.append(f"CSV File with {len(df)} rows and {len(df.columns)} columns")
        text_parts.append(f"Columns: {', '.join(df.columns.tolist())}")

        # Add column descriptions and sample data
        for col in df.columns:
            text_parts.append(f"Column '{col}' has {df[col].nunique()} unique values")

            # For text columns, add sample content
            if df[col].dtype == 'object':
                sample_values = df[col].dropna().head(10).astype(str).tolist()
                if sample_values:
                    text_parts.append(f"Sample values in {col}: {', '.join(sample_values[:5])}")
                    # Add all text data for NLP
                    text_parts.extend(df[col].dropna().astype(str).tolist())

        # Add data rows as text (first 50 rows)
        for idx, row in df.head(50).iterrows():
            row_text = f"Row {idx + 1}: "
            for col in df.columns:
                row_text += f"{col}: {str(row[col])}, "
            text_parts.append(row_text[:500])

        return {
            'text': ' '.join(text_parts),
            'structured_info': structured_info,
            'dataframe': df
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Error parsing CSV: {str(e)}")

# ============ NEW: JSON EXTRACTION FUNCTION ============
def extract_text_from_json(file_path: str) -> Dict:
    """Extract text and data from JSON file"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # Store structured data info
        structured_info = {
            'type': 'json',
            'data_type': type(data).__name__,
            'keys': list(data.keys()) if isinstance(data, dict) else None,
            'length': len(data) if hasattr(data, '__len__') else 1
        }

        # Convert to text for NLP analysis
        text_parts = []

        if isinstance(data, dict):
            text_parts.append(f"JSON Object with {len(data.keys())} keys")
            text_parts.append(f"Keys: {', '.join(data.keys())}")

            for key, value in data.items():
                text_parts.append(f"Key '{key}': {str(value)[:200]}")

                # Extract nested structures
                if isinstance(value, dict):
                    text_parts.append(f"  Nested in '{key}': {', '.join(value.keys())}")
                    for sub_key, sub_value in value.items():
                        if isinstance(sub_value, str):
                            text_parts.append(sub_value[:200])
                elif isinstance(value, list):
                    text_parts.append(f"  List in '{key}' has {len(value)} items")
                    for item in value[:10]:
                        if isinstance(item, str):
                            text_parts.append(item[:200])
                elif isinstance(value, str):
                    text_parts.append(value[:200])

        elif isinstance(data, list):
            text_parts.append(f"JSON Array with {len(data)} items")
            for idx, item in enumerate(data[:30]):
                text_parts.append(f"Item {idx + 1}: {str(item)[:200]}")
                if isinstance(item, str):
                    text_parts.append(item[:200])
                elif isinstance(item, dict):
                    text_parts.append(f"  Item keys: {', '.join(item.keys())}")
        else:
            text_parts.append(str(data))

        return {
            'text': ' '.join(text_parts),
            'structured_info': structured_info,
            'json_data': data
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Error parsing JSON: {str(e)}")

# ============ UPDATED: MAIN EXTRACTION FUNCTION ============
def extract_text_from_file(file_path: str, filename: str) -> tuple:
    """Extract text from various file formats including CSV and JSON"""
    try:
        # PDF files
        if filename.endswith('.pdf'):
            with open(file_path, 'rb') as f:
                reader = PyPDF2.PdfReader(f)
                text = ''
                for page in reader.pages:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + '\n'
                return text, None

        # DOCX files
        elif filename.endswith('.docx'):
            doc = docx.Document(file_path)
            text = '\n'.join([p.text for p in doc.paragraphs if p.text])
            return text, None

        # CSV files
        elif filename.endswith('.csv'):
            result = extract_text_from_csv(file_path)
            return result['text'], result['structured_info']

        # JSON files
        elif filename.endswith('.json'):
            result = extract_text_from_json(file_path)
            return result['text'], result['structured_info']

        # TXT files (default)
        else:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()
                return text, None

    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Error extracting text: {str(e)}")

def calculate_readability(text: str) -> float:
    """Calculate readability score (0-100)"""
    sentences = [s for s in text.split('.') if len(s.strip()) > 10]
    words = text.split()

    if len(sentences) == 0 or len(words) == 0:
        return 50.0

    avg_words_per_sentence = len(words) / len(sentences)
    score = max(0, min(100, 100 - (avg_words_per_sentence * 2)))
    return round(score, 2)

def analyze_sentiment(text: str) -> Dict:
    """Analyze sentiment using TextBlob"""
    blob = TextBlob(text[:5000])
    sentiment_score = blob.sentiment.polarity

    if sentiment_score > 0.1:
        sentiment_label = "Positive"
        sentiment_icon = "😊"
    elif sentiment_score < -0.1:
        sentiment_label = "Negative"
        sentiment_icon = "😟"
    else:
        sentiment_label = "Neutral"
        sentiment_icon = "😐"

    return {
        'score': round(sentiment_score, 3),
        'label': sentiment_label,
        'icon': sentiment_icon,
        'subjectivity': round(blob.sentiment.subjectivity, 3)
    }

def extract_entities_spacy(text: str) -> List[Dict]:
    """Extract named entities using spaCy"""
    doc = nlp(text[:10000])
    entities = []

    entity_labels = {
        'ORG': 'Organization',
        'PERSON': 'Person',
        'GPE': 'Location',
        'DATE': 'Date',
        'MONEY': 'Monetary Value',
        'LAW': 'Legal Reference',
        'EVENT': 'Event'
    }

    for ent in doc.ents:
        if ent.label_ in entity_labels:
            entities.append({
                'text': ent.text,
                'label': entity_labels.get(ent.label_, ent.label_),
                'type': ent.label_,
                'confidence': round(ent.start_char / max(len(text), 1), 2)
            })

    seen = set()
    unique_entities = []
    for e in entities:
        if e['text'] not in seen:
            seen.add(e['text'])
            unique_entities.append(e)

    return unique_entities[:20]

def extract_keywords_tfidf(text: str, top_n: int = 15) -> Dict:
    """Extract keywords using TF-IDF"""
    text_clean = re.sub(r'[^\w\s]', '', text.lower())
    words = text_clean.split()

    legal_stop_words = {'the', 'a', 'an', 'and', 'or', 'of', 'to', 'for', 'in', 'on', 'at',
                        'by', 'with', 'without', 'under', 'upon', 'shall', 'may', 'will',
                        'must', 'should', 'would', 'could', 'party', 'parties'}

    word_counts = Counter([w for w in words if w not in legal_stop_words and len(w) > 3])

    return dict(word_counts.most_common(top_n))

def detect_clauses_advanced(text: str) -> List[Dict]:
    """Advanced clause detection with confidence scoring"""
    sentences = [s.strip() for s in text.replace('\n', ' ').split('.') if len(s.strip()) > 20]
    detected_clauses = []

    for clause_type, clause_info in CLAUSE_PATTERNS.items():
        best_match = None
        best_score = 0
        best_sentence = None

        for sentence in sentences[:50]:
            sentence_lower = sentence.lower()
            score = 0
            matched_keywords = []

            for keyword in clause_info['keywords']:
                if keyword in sentence_lower:
                    score += clause_info['weight']
                    matched_keywords.append(keyword)

            if score > best_score:
                best_score = score
                best_sentence = sentence
                best_match = matched_keywords

        if best_sentence and best_score > 0:
            confidence = min(0.95, best_score / 3)
            detected_clauses.append({
                'type': clause_type,
                'text': best_sentence[:250],
                'confidence': round(confidence, 2),
                'risk_level': clause_info['risk_level'],
                'matched_keywords': best_match[:3] if best_match else []
            })

    return detected_clauses

def analyze_risks_advanced(text: str, clauses: List[Dict]) -> List[Dict]:
    """Advanced risk analysis with severity scoring"""
    text_lower = text.lower()
    risks = []

    for keyword, severity in RISK_KEYWORDS.items():
        if keyword in text_lower:
            sentences = text.split('.')
            context = ""
            for s in sentences:
                if keyword in s.lower():
                    context = s.strip()[:150]
                    break

            risk_level = "High" if severity > 0.8 else "Medium" if severity > 0.6 else "Low"

            risks.append({
                'keyword': keyword,
                'severity': severity,
                'risk_level': risk_level,
                'context': context,
                'recommendation': get_recommendation(keyword)
            })

    for clause in clauses:
        if clause['risk_level'] == 'high' and clause['confidence'] > 0.7:
            risks.append({
                'keyword': f"{clause['type']} clause",
                'severity': 0.75,
                'risk_level': 'High',
                'context': clause['text'][:150],
                'recommendation': f"Review {clause['type'].lower()} clause carefully"
            })

    seen = set()
    unique_risks = []
    for r in risks:
        if r['keyword'] not in seen:
            seen.add(r['keyword'])
            unique_risks.append(r)

    return sorted(unique_risks, key=lambda x: x['severity'], reverse=True)[:10]

def get_recommendation(keyword: str) -> str:
    """Get recommendation based on risk keyword"""
    recommendations = {
        'unlimited liability': 'Consider negotiating a liability cap',
        'no warranty': 'Request warranty provisions to protect your interests',
        'as is': 'Request inspections or warranties before acceptance',
        'no liability': 'Ensure insurance coverage for potential risks',
        'indemnify': 'Review indemnification scope and limits carefully',
        'breach': 'Understand consequences and cure periods',
        'liquidated damages': 'Verify damages are reasonable and not punitive'
    }
    return recommendations.get(keyword, 'Consult with legal counsel for review')

def generate_advanced_summary(text: str) -> str:
    """Generate better summary using key sentence extraction"""
    sentences = [s.strip() for s in text.replace('\n', ' ').split('.') if len(s.strip()) > 30]

    if not sentences:
        return text[:300] + "..."

    legal_keywords = ['agree', 'contract', 'party', 'liable', 'terminate', 'confidential',
                      'payment', 'warranty', 'indemnify', 'breach']

    scored_sentences = []
    for sentence in sentences[:30]:
        score = sum(1 for kw in legal_keywords if kw in sentence.lower())
        scored_sentences.append((score, sentence))

    scored_sentences.sort(reverse=True, key=lambda x: x[0])
    top_sentences = [s[1] for s in scored_sentences[:3]]

    summary = '. '.join(top_sentences)
    if len(summary) < 200 and len(sentences) > 3:
        summary = '. '.join(sentences[:3])

    return summary[:500] + "..."

def answer_question_smart(text: str, question: str) -> Dict:
    """Smart question answering using similarity matching"""
    question_lower = question.lower()
    question_keywords = [w for w in question_lower.split() if len(w) > 3]

    sentences = [s.strip() for s in text.replace('\n', ' ').split('.') if len(s.strip()) > 20]

    scored_sentences = []
    for sentence in sentences:
        sentence_lower = sentence.lower()
        exact_bonus = 2 if any(q in sentence_lower for q in question_keywords) else 0
        keyword_score = sum(1 for kw in question_keywords if kw in sentence_lower)
        legal_terms = ['terminate', 'liability', 'confidential', 'payment', 'warranty', 'indemnify']
        legal_bonus = sum(1 for term in legal_terms if term in sentence_lower) * 0.5

        total_score = keyword_score + exact_bonus + legal_bonus
        if total_score > 0:
            scored_sentences.append((total_score, sentence))

    scored_sentences.sort(reverse=True, key=lambda x: x[0])

    if scored_sentences:
        best_answer = scored_sentences[0][1][:400]
        confidence = min(0.95, scored_sentences[0][0] / max(len(question_keywords), 1))
        return {
            'answer': best_answer,
            'confidence': round(confidence, 2),
            'found': True
        }
    else:
        return {
            'answer': "I couldn't find a specific answer to your question. Please try rephrasing or ask about different topics like termination, liability, or payment terms.",
            'confidence': 0.2,
            'found': False
        }

# ============ UPDATED HTML WITH CSV/JSON SUPPORT ============
@app.get("/", response_class=HTMLResponse)
async def home():
    return """
    <!DOCTYPE html>
    <html>
    <head>
        <title>Enhanced Legal Document Analyzer</title>
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <style>
            * { margin: 0; padding: 0; box-sizing: border-box; }
            body {
                font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
                background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);
                min-height: 100vh;
                padding: 20px;
            }
            .container { max-width: 1400px; margin: 0 auto; }
            .header { text-align: center; color: white; margin-bottom: 30px; }
            .header h1 { font-size: 2.5em; margin-bottom: 10px; }
            .header p { font-size: 1.1em; opacity: 0.9; }
            .supported-formats {
                display: inline-block;
                background: rgba(255,255,255,0.2);
                padding: 5px 15px;
                border-radius: 20px;
                margin-top: 10px;
                font-size: 0.9em;
            }
            .card {
                background: white;
                border-radius: 20px;
                padding: 30px;
                margin-bottom: 20px;
                box-shadow: 0 10px 40px rgba(0,0,0,0.2);
            }
            .upload-area {
                border: 3px dashed #2a5298;
                border-radius: 15px;
                padding: 40px;
                text-align: center;
                cursor: pointer;
                transition: all 0.3s;
            }
            .upload-area:hover { background: #f0f4ff; border-color: #1e3c72; }
            .loading {
                display: none;
                text-align: center;
                padding: 40px;
            }
            .spinner {
                border: 4px solid #f3f3f3;
                border-top: 4px solid #2a5298;
                border-radius: 50%;
                width: 50px;
                height: 50px;
                animation: spin 1s linear infinite;
                margin: 0 auto 20px;
            }
            @keyframes spin {
                0% { transform: rotate(0deg); }
                100% { transform: rotate(360deg); }
            }
            .clause-card {
                background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
                border-left: 4px solid #2a5298;
                padding: 15px;
                margin: 10px 0;
                border-radius: 10px;
            }
            .risk-high { background: #fee; border-left-color: #dc3545; }
            .risk-medium { background: #fff3e0; border-left-color: #ffc107; }
            .risk-low { background: #e8f5e9; border-left-color: #28a745; }
            .entity-tag {
                display: inline-block;
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                color: white;
                padding: 5px 15px;
                border-radius: 20px;
                margin: 5px;
                font-size: 0.85em;
            }
            .metric {
                display: inline-block;
                background: #f8f9fa;
                padding: 15px;
                margin: 10px;
                border-radius: 10px;
                text-align: center;
                min-width: 150px;
            }
            .metric-value { font-size: 2em; font-weight: bold; color: #2a5298; }
            .btn {
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                color: white;
                border: none;
                padding: 12px 30px;
                border-radius: 25px;
                cursor: pointer;
                font-size: 1em;
            }
            input[type="text"] {
                padding: 12px;
                border: 2px solid #ddd;
                border-radius: 25px;
                width: 70%;
                margin-right: 10px;
                font-size: 1em;
            }
            .answer-card {
                background: #e8f5e9;
                border-left: 4px solid #28a745;
                padding: 15px;
                margin-top: 15px;
                border-radius: 10px;
            }
            .confidence-badge {
                display: inline-block;
                padding: 3px 10px;
                border-radius: 15px;
                font-size: 0.8em;
                margin-left: 10px;
            }
            .confidence-high { background: #28a745; color: white; }
            .confidence-medium { background: #ffc107; color: #333; }
            .confidence-low { background: #dc3545; color: white; }
            .keyword-cloud {
                display: flex;
                flex-wrap: wrap;
                gap: 10px;
                margin-top: 15px;
            }
            .keyword-item {
                background: #e3f2fd;
                padding: 8px 15px;
                border-radius: 20px;
                font-size: 0.9em;
            }
            .keyword-count { background: #2a5298; color: white; border-radius: 10px; padding: 2px 6px; margin-left: 8px; }
            hr { margin: 20px 0; border: none; border-top: 1px solid #e0e0e0; }
        </style>
    </head>
    <body>
        <div class="container">
            <div class="header">
                <h1>⚖️ Enhanced Legal Document Analyzer</h1>
                <p>AI-Powered Legal Intelligence with Machine Learning</p>
                <div class="supported-formats">
                    📄 Supported: PDF | DOCX | TXT | CSV | JSON
                </div>
            </div>

            <div class="card">
                <div class="upload-area" onclick="document.getElementById('fileInput').click()">
                    <div style="font-size: 48px; margin-bottom: 10px;">📄</div>
                    <h3>Upload Legal Document</h3>
                    <p>PDF, DOCX, TXT, CSV, or JSON files supported</p>
                    <input type="file" id="fileInput" accept=".pdf,.docx,.txt,.csv,.json" style="display: none;">
                </div>

                <div class="loading" id="loading">
                    <div class="spinner"></div>
                    <h3>Analyzing Document...</h3>
                    <p>Using advanced ML models for accurate analysis</p>
                </div>
            </div>

            <div id="results" style="display: none;"></div>
        </div>

        <script>
        let currentDocId = null;

        document.getElementById('fileInput').onchange = async function(e) {
            const file = e.target.files[0];
            if (!file) return;

            document.querySelector('.upload-area').parentElement.style.display = 'none';
            document.getElementById('loading').style.display = 'block';

            const formData = new FormData();
            formData.append('file', file);

            try {
                const uploadRes = await fetch('/upload', { method: 'POST', body: formData });
                const uploadData = await uploadRes.json();
                currentDocId = uploadData.doc_id;

                const analyzeRes = await fetch(`/analyze/${currentDocId}`);
                const data = await analyzeRes.json();

                displayResults(data);

                document.getElementById('loading').style.display = 'none';
                document.getElementById('results').style.display = 'block';
            } catch (error) {
                console.error('Error:', error);
                alert('Error processing document. Please try again.');
                location.reload();
            }
        };

        function getConfidenceClass(confidence) {
            if (confidence >= 0.7) return 'confidence-high';
            if (confidence >= 0.4) return 'confidence-medium';
            return 'confidence-low';
        }

        function displayResults(data) {
            let html = '<div class="card">';
            html += '<h2>📊 Analysis Results</h2>';
            html += `<p><strong>Document:</strong> ${data.filename}</p>`;
            html += `<p><strong>Analyzed at:</strong> ${new Date().toLocaleString()}</p>`;
            html += '<hr>';

            // Metrics
            html += '<div style="text-align: center;">';
            html += `<div class="metric"><div class="metric-value">${data.word_count}</div><div>Words</div></div>`;
            html += `<div class="metric"><div class="metric-value">${data.readability_score}</div><div>Readability</div></div>`;
            html += `<div class="metric"><div class="metric-value">${data.sentiment.icon}</div><div>${data.sentiment.label}</div></div>`;
            html += '</div><hr>';

            // Summary
            html += '<h3>📝 Executive Summary</h3>';
            html += `<p>${data.summary}</p><hr>`;

            // Key Terms
            html += '<h3>📊 Key Legal Terms</h3>';
            html += '<div class="keyword-cloud">';
            for (const [term, count] of Object.entries(data.keywords)) {
                html += `<span class="keyword-item">${term}<span class="keyword-count">${count}</span></span>`;
            }
            html += '</div><hr>';

            // Clauses
            html += '<h3>📋 Detected Clauses</h3>';
            data.clauses.forEach(clause => {
                let riskClass = clause.risk_level === 'high' ? 'risk-high' : (clause.risk_level === 'medium' ? 'risk-medium' : 'risk-low');
                html += `<div class="clause-card ${riskClass}">
                    <strong>${clause.type}</strong>
                    <span class="confidence-badge ${getConfidenceClass(clause.confidence)}">${Math.round(clause.confidence * 100)}%</span>
                    <p style="margin-top: 10px;">${clause.text}</p>
                    <small>Risk Level: ${clause.risk_level.toUpperCase()} | Matched: ${clause.matched_keywords.join(', ')}</small>
                </div>`;
            });
            html += '<hr>';

            // Entities
            html += '<h3>🏷️ Recognized Entities (spaCy)</h3>';
            data.entities.forEach(entity => {
                html += `<span class="entity-tag">${entity.text} (${entity.label})</span>`;
            });
            html += '<hr>';

            // Risks
            html += '<h3>⚠️ Risk Analysis</h3>';
            data.risks.forEach(risk => {
                let riskClass = risk.risk_level === 'High' ? 'risk-high' : (risk.risk_level === 'Medium' ? 'risk-medium' : 'risk-low');
                html += `<div class="clause-card ${riskClass}">
                    <strong>⚠️ ${risk.keyword.toUpperCase()}</strong>
                    <span class="confidence-badge">Risk: ${risk.risk_level}</span>
                    <p style="margin-top: 10px;">${risk.context || 'Found in document'}</p>
                    <small>💡 Recommendation: ${risk.recommendation}</small>
                </div>`;
            });
            html += '<hr>';

            // Q&A Section
            html += '<h3>🔍 Ask Questions About This Document</h3>';
            html += `<div style="margin-top: 15px;">
                <input type="text" id="questionInput" placeholder="Example: What are the termination conditions? Who is liable for damages?">
                <button class="btn" onclick="askQuestion()">Ask AI</button>
                <div id="answerSection"></div>
            </div>`;

            html += '</div>';
            document.getElementById('results').innerHTML = html;
        }

        async function askQuestion() {
            const question = document.getElementById('questionInput').value;
            if (!question || !currentDocId) {
                alert('Please enter a question');
                return;
            }

            const response = await fetch('/query', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({ doc_id: currentDocId, question: question })
            });
            const data = await response.json();

            const confidenceClass = data.confidence >= 0.7 ? 'confidence-high' : (data.confidence >= 0.4 ? 'confidence-medium' : 'confidence-low');

            let html = `<div class="answer-card">
                <strong>🤖 Answer:</strong>
                <span class="confidence-badge ${confidenceClass}">${Math.round(data.confidence * 100)}% confidence</span>
                <p style="margin-top: 10px;">${data.answer}</p>
            </div>`;

            document.getElementById('answerSection').innerHTML = html;
        }
        </script>
    </body>
    </html>
    """

@app.post("/upload")
async def upload_file(file: UploadFile = File(...)):
    """Upload and process document (supports PDF, DOCX, TXT, CSV, JSON)"""
    doc_id = str(uuid.uuid4())
    file_path = f"/tmp/{doc_id}_{file.filename}"

    content = await file.read()
    with open(file_path, "wb") as f:
        f.write(content)

    # Extract text based on file type
    text, structured_info = extract_text_from_file(file_path, file.filename)

    documents[doc_id] = {
        'text': text,
        'filename': file.filename,
        'uploaded_at': datetime.now().isoformat(),
        'structured_info': structured_info  # Store CSV/JSON structure info
    }

    return {"doc_id": doc_id, "filename": file.filename, "status": "success"}

@app.get("/analyze/{doc_id}")
async def analyze_document(doc_id: str):
    """Enhanced document analysis with ML"""
    if doc_id not in documents:
        raise HTTPException(status_code=404, detail="Document not found")

    doc = documents[doc_id]
    text = doc['text']

    if doc_id in analysis_cache:
        return analysis_cache[doc_id]

    # Perform analysis
    clauses = detect_clauses_advanced(text)
    entities = extract_entities_spacy(text)
    risks = analyze_risks_advanced(text, clauses)
    keywords = extract_keywords_tfidf(text)
    sentiment = analyze_sentiment(text)
    summary = generate_advanced_summary(text)
    readability = calculate_readability(text)

    result = {
        'doc_id': doc_id,
        'filename': doc['filename'],
        'summary': summary,
        'clauses': clauses,
        'entities': entities,
        'risks': risks,
        'sentiment': sentiment,
        'keywords': keywords,
        'readability_score': readability,
        'word_count': len(text.split()),
        'confidence_scores': {
            'clause_detection': round(len(clauses) / 10, 2) if clauses else 0.5,
            'entity_recognition': round(len(entities) / 20, 2) if entities else 0.5,
            'risk_assessment': round(len(risks) / 10, 2) if risks else 0.5
        }
    }

    # Add structured data info if available (for CSV/JSON)
    if doc.get('structured_info'):
        result['structured_data'] = doc['structured_info']

    analysis_cache[doc_id] = result
    return result

@app.post("/query")
async def query_document(data: dict):
    """Smart Q&A system"""
    doc_id = data.get('doc_id')
    question = data.get('question', '')

    if doc_id not in documents:
        return {"answer": "Document not found. Please upload a document first.", "confidence": 0}

    text = documents[doc_id]['text']
    result = answer_question_smart(text, question)

    return result

@app.get("/health")
async def health_check():
    """Health check endpoint"""
    return {
        "status": "healthy",
        "documents_loaded": len(documents),
        "analyzed_docs": len(analysis_cache),
        "version": "3.0",
        "features": ["spaCy NER", "TF-IDF Keywords", "Advanced Risk Analysis", "Sentiment Analysis"],
        "supported_formats": ["PDF", "DOCX", "TXT", "CSV", "JSON"]
    }

print("✅ Enhanced Legal Document Analyzer created successfully!")
print("🚀 Features: spaCy NER | TF-IDF Keywords | Advanced Risk Analysis | Sentiment Analysis")
print("📄 Supported formats: PDF, DOCX, TXT, CSV, JSON")

Overwriting legal_app.py


In [ ]:
# CELL 3: Start FastAPI server
import nest_asyncio
import uvicorn
import threading
import time

nest_asyncio.apply()

def run_server():
    uvicorn.run("legal_app:app", host="0.0.0.0", port=8000, log_level="info")

# Start server in background
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

time.sleep(3)

print("\n" + "="*60)
print("✅ ENHANCED LEGAL ANALYZER IS RUNNING!")
print("="*60)
print("\n📍 LOCAL URL: http://localhost:8000")
print("📚 API DOCS: http://localhost:8000/docs")
print("\n🚀 NEW FEATURES:")
print("   • spaCy NER for better entity recognition")
print("   • TF-IDF keyword extraction")
print("   • Advanced risk scoring")
print("   • Sentiment analysis")
print("   • Readability scoring")
print("\n" + "="*60)

INFO:     Started server process [4883]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.



✅ ENHANCED LEGAL ANALYZER IS RUNNING!

📍 LOCAL URL: http://localhost:8000
📚 API DOCS: http://localhost:8000/docs

🚀 NEW FEATURES:
   • spaCy NER for better entity recognition
   • TF-IDF keyword extraction
   • Advanced risk scoring
   • Sentiment analysis
   • Readability scoring



In [ ]:
# CELL 4: Create public URL
from google.colab.output import eval_js
from IPython.display import HTML, display

public_url = eval_js("google.colab.kernel.proxyPort(8000)")

print("\n" + "="*60)
print("🎉 YOUR ENHANCED LEGAL ANALYZER IS LIVE!")
print("="*60)
print(f"\n📍 PUBLIC URL: {public_url}")
print(f"📚 API DOCS: {public_url}docs")
print("\n" + "="*60)

display(HTML(f'''
<div style="background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);
            padding: 30px;
            border-radius: 15px;
            text-align: center;
            margin: 20px 0;">
    <h2 style="color: white;">🚀 Enhanced Legal Document Analyzer</h2>
    <a href="{public_url}" target="_blank"
       style="display: inline-block;
              background: white;
              color: #2a5298;
              padding: 15px 40px;
              text-decoration: none;
              border-radius: 50px;
              font-size: 18px;
              font-weight: bold;
              margin: 10px;">
        🔗 Click to Open App
    </a>
    <p style="color: white; margin-top: 15px;">
        📋 Features: NLP | ML Models | Risk Analysis | Q&A
    </p>
</div>
'''))

print("\n💡 UPGRADES INCLUDED:")
print("   ✅ spaCy Named Entity Recognition")
print("   ✅ TF-IDF Keyword Extraction")
print("   ✅ Advanced Risk Scoring (0-1 scale)")
print("   ✅ Sentiment Analysis with TextBlob")
print("   ✅ Readability Score Calculation")
print("   ✅ Smart Q&A with Confidence Scores")
print("   ✅ Clause Categorization with Risk Levels")
print("\n⚠️ Keep this Colab tab open to maintain the connection")


🎉 YOUR ENHANCED LEGAL ANALYZER IS LIVE!

📍 PUBLIC URL: https://8000-m-s-kkb-usc1c1-umaopmuhdgz2-c.us-central1-1.prod.colab.dev
📚 API DOCS: https://8000-m-s-kkb-usc1c1-umaopmuhdgz2-c.us-central1-1.prod.colab.devdocs




💡 UPGRADES INCLUDED:
   ✅ spaCy Named Entity Recognition
   ✅ TF-IDF Keyword Extraction
   ✅ Advanced Risk Scoring (0-1 scale)
   ✅ Sentiment Analysis with TextBlob
   ✅ Readability Score Calculation
   ✅ Smart Q&A with Confidence Scores
   ✅ Clause Categorization with Risk Levels

⚠️ Keep this Colab tab open to maintain the connection


In [ ]:
# CELL 5: Test the API
import requests

print("Testing the Legal Document Analyzer...")
print("="*50)

# Health check
try:
    response = requests.get("http://localhost:8000/health")
    print(f"✅ Health Check: {response.json()}")
except Exception as e:
    print(f"❌ Error: {e}")

print("\n🎉 System is ready to use!")

Testing the Legal Document Analyzer...
INFO:     127.0.0.1:55434 - "GET /health HTTP/1.1" 200 OK
✅ Health Check: {'status': 'healthy', 'documents_loaded': 3, 'analyzed_docs': 3, 'version': '3.0', 'features': ['spaCy NER', 'TF-IDF Keywords', 'Advanced Risk Analysis', 'Sentiment Analysis']}

🎉 System is ready to use!
